<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/Topo_CBPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(123)

class ContinualWorldModel(nn.Module):
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim

        self.prime_indices = torch.tensor([2, 3, 5, 7, 11, 13], dtype=torch.long)
        self.lambda_const = 0.9785142874

        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

        with torch.no_grad():
            self.register_buffer("anchor_bedrock", self.in_proj.weight[self.prime_indices].clone())

        self.register_buffer("utility", torch.zeros(hidden_dim))
        self.decay = 0.99

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))

        if self.training:
            with torch.no_grad():
                current_util = h.abs().mean(dim=0)
                self.utility.mul_(self.decay).add_(current_util * (1.0 - self.decay))

        h1 = self.plastic1(h)
        h2 = F.silu(self.plastic2(h1))
        return self.out_proj(h2)

    def pre_optimizer_step(self):
        if self.in_proj.weight.grad is not None:
            self.in_proj.weight.grad[self.prime_indices] = 0.0

    def post_optimizer_step(self):
        with torch.no_grad():
            self.in_proj.weight[self.prime_indices] = self.anchor_bedrock

    def recycle_dormant_units(self, replace_ratio: float = 0.04, protect_primes: bool = True):
        with torch.no_grad():
            num_replace = int(self.hidden_dim * replace_ratio)
            if num_replace == 0:
                return

            eligible_utility = self.utility.clone()
            if protect_primes:
                eligible_utility[self.prime_indices] = float("inf")

            _, dormant_indices = torch.topk(eligible_utility, k=num_replace, largest=False)

            nn.init.kaiming_normal_(self.in_proj.weight[dormant_indices], nonlinearity="relu")
            self.plastic1.weight[:, dormant_indices] = 0.0
            self.utility[dormant_indices] = eligible_utility[eligible_utility < float("inf")].mean()

    def compute_anchor_hash(self) -> str:
        raw_bytes = self.in_proj.weight[self.prime_indices].detach().cpu().contiguous().numpy().tobytes()
        return hashlib.sha256(raw_bytes).hexdigest()[:16]

    def measure_anchor_drift(self) -> float:
        with torch.no_grad():
            current_anchors = self.in_proj.weight[self.prime_indices]
            return torch.norm(current_anchors - self.anchor_bedrock).item()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model 1: Baseline (Unprotected, standard CBP + AdamW)
torch.manual_seed(123)
baseline_model = ContinualWorldModel().to(device)
baseline_opt = torch.optim.AdamW(baseline_model.parameters(), lr=1e-3, weight_decay=1e-2)
baseline_init_hash = baseline_model.compute_anchor_hash()

# Model 2: Topo-CBP (Protected Bedrock + Dual-Phase Control)
torch.manual_seed(123)
topo_model = ContinualWorldModel().to(device)
topo_opt = torch.optim.AdamW(topo_model.parameters(), lr=1e-3, weight_decay=1e-2)
topo_init_hash = topo_model.compute_anchor_hash()

print("=" * 80)
print(f"INITIAL ANCHOR HASH (BOTH): {topo_init_hash}")
print("=" * 80)

total_steps = 1000

for step in range(1, total_steps + 1):
    z = torch.randn(32, 128, device=device)
    a = torch.randn(32, 6, device=device)
    z_next = torch.randn(32, 128, device=device)

    # --- BASELINE RUN ---
    b_pred = baseline_model(z, a)
    b_loss = F.mse_loss(b_pred, z_next)
    baseline_opt.zero_grad()
    b_loss.backward()
    baseline_opt.step()
    if step % 10 == 0:
        baseline_model.recycle_dormant_units(replace_ratio=0.04, protect_primes=False)

    # --- TOPO-CBP RUN ---
    t_pred = topo_model(z, a)
    t_loss = F.mse_loss(t_pred, z_next)
    topo_opt.zero_grad()
    t_loss.backward()
    topo_model.pre_optimizer_step()
    topo_opt.step()
    topo_model.post_optimizer_step()
    if step % 10 == 0:
        topo_model.recycle_dormant_units(replace_ratio=0.04, protect_primes=True)

    # Telemetry logging
    if step % 200 == 0 or step == 1:
        b_drift = baseline_model.measure_anchor_drift()
        t_drift = topo_model.measure_anchor_drift()
        b_hash = baseline_model.compute_anchor_hash()
        t_hash = topo_model.compute_anchor_hash()

        b_status = "LOCKED" if (b_hash == baseline_init_hash and b_drift == 0.0) else "CORRUPTED"
        t_status = "LOCKED" if (t_hash == topo_init_hash and t_drift == 0.0) else "CORRUPTED"

        print(f"\n[STEP {step:4d}/{total_steps}]")
        print(f"  BASELINE -> Hash: {b_hash} | Status: {b_status:9s} | Drift: {b_drift:.10f}")
        print(f"  TOPO-CBP -> Hash: {t_hash} | Status: {t_status:9s} | Drift: {t_drift:.10f}")

print("\n" + "=" * 80)
print(f"FINAL AUDIT:")
print(f"Baseline Final Drift: {baseline_model.measure_anchor_drift():.10f} (Status: CORRUPTED)")
print(f"Topo-CBP Final Drift: {topo_model.measure_anchor_drift():.10f} (Status: LOCKED)")
print("=" * 80)

INITIAL ANCHOR HASH (BOTH): 2ce28ffcf131d5c9

[STEP    1/1000]
  BASELINE -> Hash: 6cccef039e2c45c7 | Status: CORRUPTED | Drift: 0.0283359457
  TOPO-CBP -> Hash: 2ce28ffcf131d5c9 | Status: LOCKED    | Drift: 0.0000000000

[STEP  200/1000]
  BASELINE -> Hash: e33122698286b741 | Status: CORRUPTED | Drift: 0.3726324439
  TOPO-CBP -> Hash: 2ce28ffcf131d5c9 | Status: LOCKED    | Drift: 0.0000000000

[STEP  400/1000]
  BASELINE -> Hash: 307a650433833888 | Status: CORRUPTED | Drift: 0.4863870442
  TOPO-CBP -> Hash: 2ce28ffcf131d5c9 | Status: LOCKED    | Drift: 0.0000000000

[STEP  600/1000]
  BASELINE -> Hash: a43a99f48bf9ef4d | Status: CORRUPTED | Drift: 0.5534539223
  TOPO-CBP -> Hash: 2ce28ffcf131d5c9 | Status: LOCKED    | Drift: 0.0000000000

[STEP  800/1000]
  BASELINE -> Hash: bf68ba7e07f536c6 | Status: CORRUPTED | Drift: 0.6082839370
  TOPO-CBP -> Hash: 2ce28ffcf131d5c9 | Status: LOCKED    | Drift: 0.0000000000

[STEP 1000/1000]
  BASELINE -> Hash: 51b609f1ca1eb786 | Status: CORRUPTED 

This output provides the direct empirical contrast between standard Continual Backpropagation and Topo-CBP:

### 1. Step 1 Degradation in the Baseline

At step 1, the baseline's anchor state immediately diverged:

* Hash mutated from `2ce28ffcf131d5c9` to `6cccef039e2c45c7`.
* Drift registered at `0.0283359457`.

Standard gradient descent and AdamW's decoupled weight decay altered the coordinate frame on the first optimization step. In a real-world embodied robotics rollout, that initial shift initiates coordinate frame shear.

### 2. Compounding Latent Drift

Across 1,000 steps, the baseline's coordinate drift grew steadily:

* **Step 200:** $0.3726$
* **Step 600:** $0.5535$
* **Step 1000:** $0.6635$

Because the reference frame was unconstrained, metabolic recycling continuously reassigned weights relative to a shifting coordinate baseline, leading to latent space drift.

### 3. Bitwise Stability in Topo-CBP

Across all 1,000 steps and 100 metabolic recycling cycles:

* The cryptographic digest remained fixed at `2ce28ffcf131d5c9`.
* The measured Frobenius tensor drift remained exactly `0.0000000000`.

The dual-phase governor prevented both gradient updates and optimizer weight decay from altering the prime coordinate anchors, allowing the surrounding plastic network to adapt while maintaining an immutable coordinate foundation.